In [1]:
from langgraph.graph import StateGraph,START, END
from typing import TypedDict, List,Literal
from dotenv import load_dotenv
from pydantic import BaseModel,Field
from langchain_google_genai import ChatGoogleGenerativeAI
import google.genai as genai

load_dotenv()
import os

In [2]:
# Definig the state

class QuadState(TypedDict):
    a: int
    b: int
    c: int

    equation: str
    discriminant: float
    result : str

In [3]:
def show_equation(state:QuadState):
    a = state['a']
    b = state['b']
    c = state['c']
    equation = f"{a}x^2 + {b}x + {c} = 0"
    
    return {'equation': equation}

In [4]:
def calculate_discriminant(state:QuadState):
    a = state['a']
    b = state['b']
    c = state['c']
    discriminant = b**2 - 4*a*c
    
    return {'discriminant': discriminant}

In [8]:
def real_roots(state:QuadState):
    a = state['a']
    b = state['b']
    c = state['c']
    D = state['discriminant']
    
    root1 = (-b + D**0.5) / (2*a)
    root2 = (-b - D**0.5) / (2*a)
    
    return {'result': f"Real roots: {root1}, {root2}"}

In [11]:
def repeated_roots(state:QuadState):
    a = state['a']
    b = state['b']
    c = state['c']
    D = state['discriminant']
    
    root = -b / (2*a)
    
    return {'result': f"Repeated root: {root}"}

In [12]:
def no_real_roots(state:QuadState):
    return {'result' : "No real roots exist."}

In [13]:
# THIS IS IMP
# Adding conditional branches based on discriminant value
# THIS IS THE FUNCTION WHERE OUTPUT WILL BE THE NAME OF THE NEXT NODE OR NEXT FUNCTION TO BE EXECUTED

def check_condition(state:QuadState)->Literal['real_roots','repeated_roots','no_real_roots']:
     D = state['discriminant']

     if D > 0:
         return 'real_roots'
     elif D == 0:
        return 'repeated_roots'
     else:
        return 'no_real_roots'

In [ ]:
graph = StateGraph(QuadState)

# DEFINING THE WORKFLOW NODES
graph.add_node('show_equation',show_equation)
graph.add_node('calculate_discriminant',calculate_discriminant)

graph.add_node('real_roots',real_roots)
graph.add_node('repeated_roots',repeated_roots)
graph.add_node('no_real_roots',no_real_roots)

# ADDING EDGES

graph.add_edge(START,'show_equation')
graph.add_edge('show_equation','calculate_discriminant')

# CONDITIONAL BRANCHING EDGE
# THIS WILL CALL THE FUNCTION check_condition TO DECIDE THE NEXT NODE TO EXECUTE
graph.add_conditional_edges('calculate_discriminant',check_condition) 


graph.add_edge('real_roots',END)
graph.add_edge('repeated_roots',END)    
graph.add_edge('no_real_roots',END)

workflow = graph.compile()
print(workflow.get_graph().draw_ascii())

                             +-----------+                               
                             | __start__ |                               
                             +-----------+                               
                                   *                                     
                                   *                                     
                                   *                                     
                           +---------------+                             
                           | show_equation |                             
                           +---------------+                             
                                   *                                     
                                   *                                     
                                   *                                     
                      +------------------------+                         
                      | calculate_disc

In [16]:
initial_state = {'a': 4, 'b': 2, 'c': 2}

workflow.invoke(initial_state)

{'a': 4,
 'b': 2,
 'c': 2,
 'equation': '4x^2 + 2x + 2 = 0',
 'discriminant': -28,
 'result': 'No real roots exist.'}